In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import BallTree
from scipy.optimize import minimize

# --------------------------
# CONFIG
# --------------------------
TRAIN_PATH = "binaaz_train.csv"
TEST_PATH = "binaaz_test.csv"
LANDMARKS_PATH = "baku_landmarks_clean.csv"  

RANDOM_STATE = 42
N_SPLITS = 5

# Train-only coordinate bbox filter (helps remove garbage coords)
LAT_MIN, LAT_MAX = 36, 42
LON_MIN, LON_MAX = 44, 52

# Baku center (for a general distance feature)
BAKU_CENTER_LAT, BAKU_CENTER_LON = 40.4093, 49.8671

# Fallback grouping for district if regex fails
GROUP_COL_FALLBACK = "seher"

# Keep category trees only if category appears >= this many times
MIN_CAT_COUNT = 8

# Optional models
HAVE_XGB, HAVE_LGBM = True, True
try:
    from xgboost import XGBRegressor
except Exception:
    HAVE_XGB = False

try:
    from lightgbm import LGBMRegressor
except Exception:
    HAVE_LGBM = False


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def parse_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", " ")
    m = re.findall(r"[-+]?\d*\.?\d+", s)
    if not m:
        return np.nan
    try:
        return float(m[0])
    except Exception:
        return np.nan

def parse_int_first(x):
    if pd.isna(x):
        return np.nan
    m = re.findall(r"\d+", str(x))
    return float(m[0]) if m else np.nan

def parse_floor_pair(x):
    if pd.isna(x):
        return (np.nan, np.nan)
    nums = re.findall(r"\d+", str(x))
    if len(nums) >= 2:
        return (float(nums[0]), float(nums[1]))
    if len(nums) == 1:
        return (np.nan, float(nums[0]))
    return (np.nan, np.nan)

def to_binary_yesno(x):
    if pd.isna(x):
        return 0
    s = str(x).strip().lower()
    yes_vals = {"1", "true", "yes", "var", "mövcuddur", "movcuddur", "bəli", "beli", "hə", "he"}
    return 1 if s in yes_vals else 0

def clean_text_group(s):
    s = s.astype(str).fillna("unknown").str.strip().str.lower()
    s = s.replace({"nan": "unknown", "none": "unknown", "": "unknown"})
    return s

def extract_district_from_locations(loc):
    # best-style: capture "Yasamal r." from locations text
    if pd.isna(loc):
        return np.nan
    m = re.search(r"(\w+\s*r\.)", str(loc))
    return m.group(1) if m else np.nan

def freq_encode(train_col: pd.Series, test_col: pd.Series):
    vc = train_col.value_counts(dropna=False)
    return train_col.map(vc).fillna(0).astype(float), test_col.map(vc).fillna(0).astype(float)

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Vectorized Haversine distance in km.
    lat/lon in degrees.
    """
    R = 6371.0
    lat1 = np.deg2rad(lat1); lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2); lon2 = np.deg2rad(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
landmarks = pd.read_csv(LANDMARKS_PATH)

# ------------------------------------------------------------
# Preprocess
# ------------------------------------------------------------
def preprocess(df, is_train: bool):
    df = df.copy()

    if is_train:
        df["price"] = df["price"].apply(parse_numeric)
        df = df.dropna(subset=["price"])
        df = df[df["price"] > 0]

    df["area_numeric"] = df["Sahə"].apply(parse_numeric)
    df = df.dropna(subset=["area_numeric"])
    df = df[df["area_numeric"] > 0]

    floors = df["Mərtəbə"].apply(parse_floor_pair)
    df["current_floor"] = floors.apply(lambda t: t[0])
    df["total_floors"] = floors.apply(lambda t: t[1])

    df["room_count"] = df["Otaq sayı"].apply(parse_int_first)

    # district from locations string if possible
    if "locations" in df.columns:
        df["district"] = df["locations"].apply(extract_district_from_locations)
    else:
        df["district"] = np.nan

    # fallback district = seher
    if GROUP_COL_FALLBACK in df.columns:
        df[GROUP_COL_FALLBACK] = clean_text_group(df[GROUP_COL_FALLBACK])
    else:
        df[GROUP_COL_FALLBACK] = "unknown"

    df["district"] = df["district"].fillna(df[GROUP_COL_FALLBACK]).fillna("unknown")
    df["district"] = clean_text_group(df["district"])

    # boolean fields
    df["has_kupca"] = df["Kupça"].apply(to_binary_yesno) if "Kupça" in df.columns else 0
    df["has_mortgage"] = df["İpoteka"].apply(to_binary_yesno) if "İpoteka" in df.columns else 0

    # coords
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

    # text lengths
    df["title_len"] = df["title"].astype(str).fillna("").str.len() if "title" in df.columns else 0
    df["desc_len"]  = df["description"].astype(str).fillna("").str.len() if "description" in df.columns else 0

    # optional category-like col
    df["poster_type"] = clean_text_group(df["poster_type"]) if "poster_type" in df.columns else "unknown"
    df["seher"] = clean_text_group(df["seher"]) if "seher" in df.columns else "unknown"

    return df

train = preprocess(train, True)
test = preprocess(test, False)

# ------------------------------------------------------------
# Log transform + trimming (best-result style)
# ------------------------------------------------------------
train["Log_Price"] = np.log(train["price"])
train["Log_Area"] = np.log(train["area_numeric"])

lp_lo, lp_hi = train["Log_Price"].quantile(0.01), train["Log_Price"].quantile(0.99)
la_lo, la_hi = train["Log_Area"].quantile(0.01), train["Log_Area"].quantile(0.99)

train = train[
    train["Log_Price"].between(lp_lo, lp_hi) &
    train["Log_Area"].between(la_lo, la_hi)
].copy()

# ------------------------------------------------------------
# Coordinate bbox filter (train only)
# ------------------------------------------------------------
train = train[
    train["latitude"].between(LAT_MIN, LAT_MAX) &
    train["longitude"].between(LON_MIN, LON_MAX)
].copy()

# ------------------------------------------------------------
# Impute lat/lon by district median + global fallback
# ------------------------------------------------------------
for col in ["latitude", "longitude"]:
    global_med = train[col].median()
    train[col] = train[col].fillna(train.groupby("district")[col].transform("median")).fillna(global_med)
    test[col]  = test[col].fillna(test.groupby("district")[col].transform("median")).fillna(global_med)

# ------------------------------------------------------------
# Encode district + frequency encodings (description-style)
# ------------------------------------------------------------
le_dist = LabelEncoder()
train["district_encoded"] = le_dist.fit_transform(train["district"].fillna("unknown"))
dist_map = {cls: i for i, cls in enumerate(le_dist.classes_)}
test["district_encoded"] = test["district"].map(dist_map).fillna(-1).astype(int)

train["district_freq"], test["district_freq"] = freq_encode(train["district"], test["district"])
train["seher_freq"],    test["seher_freq"]    = freq_encode(train["seher"], test["seher"])
train["poster_freq"],   test["poster_freq"]   = freq_encode(train["poster_type"], test["poster_type"])

# ------------------------------------------------------------
# Build BallTrees per category (Haversine) from landmarks :contentReference[oaicite:2]{index=2}
# ------------------------------------------------------------
landmarks = landmarks.copy()
landmarks["latitude"] = pd.to_numeric(landmarks["latitude"], errors="coerce")
landmarks["longitude"] = pd.to_numeric(landmarks["longitude"], errors="coerce")
landmarks["category"] = landmarks["category"].astype(str).fillna("unknown").str.strip().str.lower()
landmarks = landmarks.dropna(subset=["latitude", "longitude", "category"]).reset_index(drop=True)

cat_counts = landmarks["category"].value_counts()
KEEP_CATS = cat_counts[cat_counts >= MIN_CAT_COUNT].index.tolist()

cat_trees = {}
for cat in KEEP_CATS:
    sub = landmarks[landmarks["category"] == cat]
    coords_rad = np.deg2rad(sub[["latitude", "longitude"]].values)
    cat_trees[cat] = BallTree(coords_rad, metric="haversine")

def add_landmark_category_distances(df):
    df = df.copy()
    coords_rad = np.deg2rad(df[["latitude", "longitude"]].values)
    for cat, tree in cat_trees.items():
        dist_rad, _ = tree.query(coords_rad, k=1)
        df[f"dist_nearest_{cat}_km"] = dist_rad[:, 0] * 6371.0
    return df

train = add_landmark_category_distances(train)
test  = add_landmark_category_distances(test)

# ------------------------------------------------------------
# Feature engineering
# ------------------------------------------------------------
# log area for test
test["Log_Area"] = np.log(test["area_numeric"].clip(lower=1e-6))

# floor ratio
train["floor_ratio"] = (train["current_floor"] / train["total_floors"]).replace([np.inf, -np.inf], np.nan)
test["floor_ratio"]  = (test["current_floor"]  / test["total_floors"]).replace([np.inf, -np.inf], np.nan)
train["floor_ratio"] = train["floor_ratio"].fillna(0.5).clip(0, 1)
test["floor_ratio"]  = test["floor_ratio"].fillna(0.5).clip(0, 1)

# area per room
train["area_per_room"] = (train["area_numeric"] / train["room_count"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
test["area_per_room"]  = (test["area_numeric"]  / test["room_count"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
apr_med = train["area_per_room"].median()
train["area_per_room"] = train["area_per_room"].fillna(apr_med)
test["area_per_room"]  = test["area_per_room"].fillna(apr_med)

train["log_area_per_room"] = np.log(train["area_per_room"].clip(lower=1e-6))
test["log_area_per_room"]  = np.log(test["area_per_room"].clip(lower=1e-6))

# Haversine distance to Baku center (km)
train["dist_center_km"] = haversine_km(train["latitude"], train["longitude"], BAKU_CENTER_LAT, BAKU_CENTER_LON)
test["dist_center_km"]  = haversine_km(test["latitude"],  test["longitude"],  BAKU_CENTER_LAT, BAKU_CENTER_LON)

# interactions
train["floor_x_area"] = train["floor_ratio"] * train["Log_Area"]
test["floor_x_area"]  = test["floor_ratio"]  * test["Log_Area"]
train["metro_x_area"] = train.get("dist_nearest_metro_km", 0) * train["Log_Area"]
test["metro_x_area"]  = test.get("dist_nearest_metro_km", 0) * test["Log_Area"]

# ------------------------------------------------------------
# Feature list
# ------------------------------------------------------------
cat_dist_features = [f"dist_nearest_{cat}_km" for cat in KEEP_CATS]

FEATURES = [
    "Log_Area", "room_count", "current_floor", "total_floors",
    "has_kupca", "has_mortgage",
    "latitude", "longitude",
    "floor_ratio",
    "district_encoded",
    "district_freq", "seher_freq", "poster_freq",
    "area_per_room", "log_area_per_room",
    "dist_center_km", "floor_x_area", "metro_x_area",
    "title_len", "desc_len",
] + cat_dist_features

# fill missing numerics
for f in FEATURES:
    train[f] = pd.to_numeric(train[f], errors="coerce")
    train[f] = train[f].fillna(train[f].median())

train_medians = {f: train[f].median() for f in FEATURES}
for f in FEATURES:
    test[f] = pd.to_numeric(test[f], errors="coerce")
    test[f] = test[f].fillna(train_medians[f])

X = train[FEATURES].copy()
y = train["Log_Price"].copy()
groups = train["district_encoded"].values

# ------------------------------------------------------------
# Models (ensemble)
# ------------------------------------------------------------
models = {}

models["rf1"] = RandomForestRegressor(
    n_estimators=700, max_features="sqrt", min_samples_leaf=1,
    random_state=RANDOM_STATE, n_jobs=-1
)
models["rf2"] = RandomForestRegressor(
    n_estimators=1100, max_features=0.7, min_samples_leaf=2,
    random_state=RANDOM_STATE + 1, n_jobs=-1
)
models["rf3"] = RandomForestRegressor(
    n_estimators=1400, max_features="sqrt", min_samples_leaf=3,
    random_state=RANDOM_STATE + 2, n_jobs=-1
)

if HAVE_XGB:
    models["xgb"] = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

if HAVE_LGBM:
    models["lgbm"] = LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

model_names = list(models.keys())
print("Models used:", model_names)
print("Landmark categories used:", KEEP_CATS)

# ------------------------------------------------------------
# OOF predictions with GroupKFold
# ------------------------------------------------------------
n_splits = min(N_SPLITS, len(np.unique(groups)))
gkf = GroupKFold(n_splits=n_splits)

oof = {name: np.zeros(len(X)) for name in model_names}

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr = y.iloc[tr_idx]

    for name in model_names:
        m = models[name]
        m.fit(X_tr, y_tr)
        oof[name][va_idx] = m.predict(X_va)

oof_mat = np.column_stack([oof[n] for n in model_names])
y_vec = y.values

# weight optimization (optional)
def mae_loss(w):
    w = np.clip(w, 0, 1)
    w = w / (w.sum() + 1e-12)
    return mean_absolute_error(y_vec, oof_mat @ w)

w0 = np.ones(oof_mat.shape[1]) / oof_mat.shape[1]
res = minimize(mae_loss, w0, method="Nelder-Mead")
w_opt = np.clip(res.x, 0, 1)
w_opt = w_opt / (w_opt.sum() + 1e-12)
print("Optimized weights:", dict(zip(model_names, np.round(w_opt, 4))))

# Ridge stacking
stacker = Ridge(alpha=1.0, random_state=RANDOM_STATE)
stacker.fit(oof_mat, y_vec)
print("OOF log-MAE (stacked):", mean_absolute_error(y_vec, stacker.predict(oof_mat)))

# ------------------------------------------------------------
# Fit full models and predict test
# ------------------------------------------------------------
X_test = test[FEATURES].copy()

test_base_preds = []
for name in model_names:
    models[name].fit(X, y)
    test_base_preds.append(models[name].predict(X_test))

test_mat = np.column_stack(test_base_preds)

# choose final prediction method
pred_log = stacker.predict(test_mat)     # usually best
# pred_log = test_mat @ w_opt            # alternative: optimized blend

# clip log preds
lo, hi = np.quantile(y, 0.01), np.quantile(y, 0.99)
pred_log = np.clip(pred_log, lo, hi)

pred_price = np.exp(pred_log)

# final clipping in price space (optional)
p_lo, p_hi = np.quantile(train["price"].values, 0.01), np.quantile(train["price"].values, 0.99)
pred_price = np.clip(pred_price, p_lo, p_hi)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
submission = pd.DataFrame({"_id": test["_id"], "price": pred_price})
submission.to_csv("predictions3.csv", index=False)
print("\n✅ Saved predictions3.csv with columns: _id, price")


Models used: ['rf1', 'rf2', 'rf3', 'xgb', 'lgbm']
Landmark categories used: ['other', 'religious', 'historical', 'metro', 'museum', 'park']
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007251 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4149
[LightGBM] [Info] Number of data points in the train set: 51083, number of used features: 26
[LightGBM] [Info] Start training from score 11.946988
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008324 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4152
[LightGBM] [Info] Number of data points in the train set: 54561, number of used features: 26
[LightGBM] [Info] Start training from score 11.949791
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010320 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [